# X1-binário — mesma grade, alvo binário

Réplica de [experiment_1_does_ssl_helps.ipynb](experiment_1_does_ssl_helps.ipynb) (X1-multiclasse) com uma única mudança de desenho: os modelos são treinados desde o início para segmentação **binária** (fundo vs. estrutura, `target_variant='binary_c2_c4'`, `n_classes=2`), em vez de multiclasse (fundo/C2/C4, `n_classes=4`).

Fold, orçamentos, pool não rotulado, sementes e hiperparâmetros são idênticos ao X1 — ver `configs/experiment/x1-binario/`, que espelha `configs/experiment/x1/` trocando só `n_classes` e `target_variant`. Os resultados vão para `logs/x1-binario/results_per_sample.csv`, separado do X1-multiclasse.

**Por que isso existe, e não só a métrica `C2+C4 (macro)` do X1-multiclasse:** aquela métrica é a média de `C2` e `C4` calculados separadamente sobre o argmax de um modelo *multiclasse* — sensível, por construção, a confusão entre as duas estruturas (ver a célula de ressalvas naquele notebook). `FG` aqui não tem essa confusão para começo, porque nunca existiu uma segunda estrutura para confundir. Uma diferença `FG` vs. `C2+C4 (macro)` mistura dois efeitos (treinar binário desde o início + ausência de confusão C2↔C4) — ver a célula de ressalvas abaixo antes de interpretar essa comparação.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys
import torchmetrics

In [ ]:
import sys
sys.path.append("../../")

import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.data.vfss_frame_dataset import (
    load_video_frame_dataframe,
    split_by_group,
    apply_label_regime,
    build_unlabeled_frame_pool,
)

## Parâmetros

`SEEDS` varia **apenas** a estocasticidade de treino (inicialização, ordem dos batches,
augmentation). `SPLIT_SEED` e `LABEL_SEED` ficam **fixos de propósito**:

- mexer em `SPLIT_SEED` trocaria o fold, e passaríamos a medir variância de split em vez do efeito
  do orçamento;
- mexer em `LABEL_SEED` quebraria o **aninhamento** dos regimes (todo paciente revelado em 10%
  continua revelado em 20%, 50% e 100%), que é o que torna os quatro orçamentos diretamente
  comparáveis. Ver a docstring de `apply_label_regime`.

In [ ]:
DATASET_PATH = "/data_ssd/caioseda/data/dataset_inca_all_frames/"

# --- o fold (fixo) ---
SPLIT_SEED = 42
SPLIT_RATIOS = (0.7, 0.10, 0.2)
SPLIT_GROUP_COLUMN = "paciente_id"

# --- regime de rotulos (so os frames ANOTADOS) ---
LABEL_SEED = 42            # FIXO: garante o aninhamento entre orcamentos
LABEL_GROUP_COLUMN = "paciente_id"
LABEL_FRACTIONS = [0.25, 0.50, 0.75, 1.00]

# --- pool nao rotulado REAL (frames que nunca foram anotados) ---
# Identico em todos os orcamentos: e o que permite medir o ganho do SSL tambem em 100%.
UNLABELED_POOL_SIZE = 8000
UNLABELED_POOL_FRAME_STRIDE = 3   # frames consecutivos de VFSS sao quase identicos
UNLABELED_POOL_SEED = 42
UNLABELED_POOL_SPLITS = ["train"] # nenhum frame de paciente de val/test entra no treino

# --- politica de amostragem dos nao rotulados (DEFAULT do X1) ---
# Cada slot nao rotulado vem do mesmo video de um dos rotulados do batch, em qualquer posicao.
# Alternativas ('global', 'temporal_window') sao comparadas no X1B, nao aqui.
UNLABELED_POLICY = "same_video"

# --- grade ---
SEEDS = [42]               # lista; comeca com um valor so
METHODS = ["supervised", "fixmatch", "meanteacher", "diffrect"]
# DiffRect custa ~5.4x um passo supervisionado (medido: 0.243 s/passo contra 0.045, RTX 6000
# Ada, batch 6 a 256x256), entao a grade com ele leva ~5x mais tempo. Antes de interpretar uma
# celula fraca do DiffRect, conferir se experiment_acdc_ssl_reproduction.ipynb passou: la o
# metodo tem gabarito publicado, aqui nao.

# --- treino ---
MAX_STEPS = 8000           # orcamento de compute FIXO: todo run ve o mesmo numero de imagens
VAL_EVERY_N_STEPS = 500    # convertido em `check_val_every_n_epoch` por orcamento
BATCH_SIZE = 8
LABELED_BATCH_SIZE = 4     # os outros 4 vem do fluxo nao rotulado, em TODOS os orcamentos

# --- avaliacao ---
CLASS_IDS = {1: "FG"}   # 0=fundo, 1=estrutura (C2+C4 fundidos por target_variant=binary_c2_c4)
NUM_CLASSES = 2
METRICS = ["dice", "iou", "assd", "hd95"]

# Caminhos absolutos: o notebook roda a partir do proprio diretorio, e o Experiment resolveria
# `logs/` relativo a ele -- espalhando checkpoints dentro de notebooks/.
REPO_ROOT = Path("../../").resolve()
CONFIG_DIR = REPO_ROOT / "configs/experiment/x1-binario"
LOGS_DIR = REPO_ROOT / "logs"
RESULTS_DIR = LOGS_DIR / "x1-binario"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_CSV = RESULTS_DIR / "results_per_sample.csv"

---

# Parte 1 — O fold

Reproduz aqui, em pandas, exatamente a divisão que o `VFSSFrameDatasetBase` faz internamente
(mesmas funções, mesmas sementes), para poder auditá-la antes de treinar qualquer coisa.

In [ ]:
df = load_video_frame_dataframe(DATASET_PATH)
df = split_by_group(df, SPLIT_GROUP_COLUMN, seed=SPLIT_SEED, ratios=SPLIT_RATIOS)

split_summary = (
    df.groupby("split")
      .agg(frames=("video_frame", "size"),
           pacientes=("paciente_id", "nunique"),
           videos=("video_id", "nunique"))
      .reindex(["train", "val", "test"])
)
split_summary["% frames"] = (100 * split_summary.frames / len(df)).round(1)
print(f"Fold unico: split_seed={SPLIT_SEED}, agrupado por {SPLIT_GROUP_COLUMN}, ratios={SPLIT_RATIOS}")
split_summary

### Regimes de rótulo

Nenhuma linha é descartada: o que muda entre os orçamentos é só a coluna `is_labeled`. O total de
frames de treino é idêntico nos quatro — o que muda é quanta supervisão eles carregam.

In [ ]:
regime_rows = []
labeled_patients_by_fraction = {}

for fraction in LABEL_FRACTIONS:
    regime_df = apply_label_regime(
        df,
        group_column=LABEL_GROUP_COLUMN,
        label_fraction=fraction,
        seed=LABEL_SEED,
        label_regime_splits=("train",),
    )
    train_df = regime_df[regime_df.split == "train"]
    labeled = train_df[train_df.is_labeled]
    labeled_patients_by_fraction[fraction] = set(labeled.paciente_id)

    regime_rows.append({
        "orcamento": f"{fraction:.0%}",
        "anotados (treino)": len(train_df),
        "rotulos revelados": len(labeled),
        "% revelado": round(100 * len(labeled) / len(train_df), 1),
        "rotulos escondidos": len(train_df) - len(labeled),
        "pacientes revelados": labeled.paciente_id.nunique(),
        "videos revelados": labeled.video_id.nunique(),
    })

regime_summary = pd.DataFrame(regime_rows).set_index("orcamento")
regime_summary

### O pool não rotulado

O que a literatura de SSL não tem: frames que **nunca foram anotados**. São a maior parte do
dataset, e entram no treino em todos os orçamentos.

O pool é construído só a partir de vídeos do split de **treino** — um frame não rotulado de um
paciente de teste seria vazamento tão real quanto um rótulo.

In [ ]:
pool_df = build_unlabeled_frame_pool(
    DATASET_PATH,
    df,
    splits=UNLABELED_POOL_SPLITS,
    pool_size=UNLABELED_POOL_SIZE,
    frame_stride=UNLABELED_POOL_FRAME_STRIDE,
    seed=UNLABELED_POOL_SEED,
)

# Quantos frames existiriam sem stride nem teto, para dimensionar o quanto estamos deixando na mesa.
pool_maximo = build_unlabeled_frame_pool(DATASET_PATH, df, splits=UNLABELED_POOL_SPLITS)

train_anotados = int((df.split == "train").sum()) if "split" in df else None
print(f"Frames anotados no dataset inteiro : {len(df)}")
print(f"Frames nao anotados elegiveis      : {len(pool_maximo)}  (videos de {UNLABELED_POOL_SPLITS})")
print(f"Pool efetivamente usado            : {len(pool_df)}  "
      f"(stride={UNLABELED_POOL_FRAME_STRIDE}, teto={UNLABELED_POOL_SIZE}, seed={UNLABELED_POOL_SEED})")
print(f"Videos representados no pool       : {pool_df.video_id.nunique()}")
print(f"Pacientes representados no pool    : {pool_df.paciente_id.nunique()}")
print(f"\nRazao nao rotulado : rotulado no batch = "
      f"{BATCH_SIZE - LABELED_BATCH_SIZE}:{LABELED_BATCH_SIZE} (fixa em todos os orcamentos)")
print(f"Cada frame do pool e visto ~{MAX_STEPS * (BATCH_SIZE - LABELED_BATCH_SIZE) / len(pool_df):.1f}x "
      f"ao longo dos {MAX_STEPS} passos.")

### Auditoria do desenho

Três invariantes que precisam valer para o experimento significar o que se propõe. Se qualquer um
falhar, a comparação entre orçamentos está contaminada.

In [ ]:
# 1. Aninhamento: todo paciente revelado num orcamento menor continua revelado no maior.
for smaller, larger in zip(LABEL_FRACTIONS, LABEL_FRACTIONS[1:]):
    assert labeled_patients_by_fraction[smaller] <= labeled_patients_by_fraction[larger], (
        f"Aninhamento quebrado entre {smaller:.0%} e {larger:.0%} -- LABEL_SEED variou?"
    )

# 2. Numero de imagens vistas constante: os quatro orcamentos tem o mesmo total de frames anotados
#    de treino (o que muda e quantos deles carregam rotulo) e o mesmo pool nao rotulado.
assert regime_summary["anotados (treino)"].nunique() == 1, "O total de frames anotados mudou entre orcamentos."

# 3. Test set intocado pelo regime: o mascaramento so se aplica ao split de treino.
for fraction in LABEL_FRACTIONS:
    regime_df = apply_label_regime(df, group_column=LABEL_GROUP_COLUMN, label_fraction=fraction,
                                   seed=LABEL_SEED, label_regime_splits=("train",))
    assert regime_df[regime_df.split == "test"].is_labeled.all(), "Test set foi afetado pelo regime."

# 4. O pool nao vaza: nenhum frame nao rotulado vem de paciente de val/test, e nenhum frame
#    anotado aparece duas vezes (uma como anotado, outra como "nao rotulado").
pacientes_treino = set(df[df.split == "train"].paciente_id)
assert set(pool_df.paciente_id) <= pacientes_treino, "O pool contem pacientes fora do split de treino."
assert not (set(zip(pool_df.video_id, pool_df.frame_id)) & set(zip(df.video_id, df.frame_id))), (
    "Um frame anotado apareceu tambem no pool nao rotulado."
)

# 5. Ha fluxo nao supervisionado em TODOS os orcamentos -- inclusive 100%. E o que distingue este
#    desenho do da literatura, e o que faz a celula de 100% medir alguma coisa.
assert len(pool_df) > 0, "Sem pool, o orcamento de 100% nao tem dados nao rotulados e o SSL degenera."

print("OK - aninhamento, imagens vistas constantes, test set intocado, pool sem vazamento.")
print(f"    pacientes revelados por orcamento: "
      f"{ {f'{f:.0%}': len(p) for f, p in labeled_patients_by_fraction.items()} }")
print(f"    pool nao rotulado (todos os orcamentos): {len(pool_df)} frames")

### O que a política `same_video` implica

Sob `same_video`, um frame não rotulado só pode entrar no batch se o vídeo dele tiver um frame
rotulado ali — ou seja, se for de um **paciente revelado**. Isso acopla o orçamento de anotação à
cobertura do fluxo não rotulado: quanto menos rótulos, menos vídeos podem contribuir dados não
rotulados.

Não é um bug, é a consequência de avaliar os métodos num cenário que preserva a estrutura de vídeo.
Mas muda a leitura da curva do X1 — parte do ganho do SSL em orçamentos altos pode vir de ter mais
*vídeos* disponíveis, não de ter mais *rótulos*. A tabela abaixo quantifica o efeito, e é a
motivação direta do [X1B](experiment_1b_unlabeled_sampling_policy.ipynb).

In [ ]:
cobertura_rows = []
for fraction in LABEL_FRACTIONS:
    regime_df = apply_label_regime(df, group_column=LABEL_GROUP_COLUMN, label_fraction=fraction,
                                   seed=LABEL_SEED, label_regime_splits=("train",))
    revelados = regime_df[(regime_df.split == "train") & regime_df.is_labeled]
    videos_revelados = set(revelados.video_id)

    # `same_video`: so vídeos com rotulo revelado podem contribuir frames nao rotulados.
    alcancavel = pool_df[pool_df.video_id.isin(videos_revelados)]
    cobertura_rows.append({
        "orcamento": f"{fraction:.0%}",
        "videos com rotulo": len(videos_revelados),
        "pacientes com rotulo": revelados.paciente_id.nunique(),
        "pool alcancavel (same_video)": len(alcancavel),
        "% do pool alcancavel": round(100 * len(alcancavel) / len(pool_df), 1),
        "pool alcancavel (global)": len(pool_df),
    })

cobertura = pd.DataFrame(cobertura_rows).set_index("orcamento")
display(cobertura)
print("Sob `global` a ultima coluna valeria para todos os orcamentos. A diferenca entre as duas e")
print("exatamente o que o X1B mede -- e por que o orcamento principal do X1B e 100%, onde ela some.")

> ⚠️ **Ressalvas a carregar para a leitura dos resultados.**
>
> 1. **Poucos grupos.** Com agrupamento por paciente, cada orçamento revela um punhado de
>    pacientes; a curva carrega a variância de *quais* pacientes caíram, não só de *quantos*
>    rótulos existem. Trocar `LABEL_GROUP_COLUMN` para `"video_id"` dá praticamente a mesma
>    contagem de frames rotulados com muito mais diversidade de pacientes — é uma linha, e vale
>    re-rodar como checagem de robustez se alguma conclusão ficar apertada. Com `SEEDS = [42]`
>    ainda não há barra de erro nenhuma: uma única semente não sustenta um veredito.
> 2. **Deriva de distribuição do pool.** Os frames anotados foram escolhidos por alguém,
>    provavelmente em fases específicas da deglutição. O pool inclui frames em que C2/C4 estão
>    ocultos ou o contraste ainda não chegou — exatamente a condição em que pseudo-rótulos
>    degradam. Antes de culpar o método, olhe `train/pseudo_*` e `train/teacher_*` no
>    TensorBoard: são medidos contra a ground truth que o regime escondeu.
> 3. **Cobertura acoplada ao orçamento.** Com a política `same_video`, orçamentos baixos restringem
>    o fluxo não rotulado aos vídeos dos pacientes revelados (ver a auditoria acima). A curva do X1
>    mistura, portanto, "mais rótulos" com "mais vídeos contribuindo dados não rotulados". O X1B
>    separa as duas coisas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))

x = np.arange(len(LABEL_FRACTIONS))
labels = [f"{f:.0%}" for f in LABEL_FRACTIONS]

axes[0].bar(x, regime_summary["rotulos revelados"], label="rotulo revelado", color="#4C72B0")
axes[0].bar(x, regime_summary["rotulos escondidos"], bottom=regime_summary["rotulos revelados"],
            label="rotulo escondido", color="#AAAAAA")
axes[0].bar(x, [len(pool_df)] * len(x),
            bottom=regime_summary["anotados (treino)"],
            label="pool (nunca anotado)", color="#DDDDDD")
axes[0].set_xticks(x); axes[0].set_xticklabels(labels)
axes[0].set_ylabel("frames de treino")
axes[0].set_title("Composicao do treino\n(pool constante em todos os orcamentos)")
axes[0].legend(frameon=False, fontsize=8)

axes[1].plot(x, regime_summary["pacientes revelados"], "o-", label="pacientes")
axes[1].plot(x, regime_summary["videos revelados"], "s-", label="videos")
axes[1].set_xticks(x); axes[1].set_xticklabels(labels)
axes[1].set_ylabel("grupos revelados")
axes[1].set_title("Diversidade da supervisao")
axes[1].legend(frameon=False, fontsize=8)

for ax in axes:
    ax.set_xlabel("orcamento de anotacao")
    ax.spines[["top", "right"]].set_visible(False)
plt.tight_layout()

---

# Parte 2 — Treino e avaliação

A grade é `semente × método × orçamento`. Três escolhas de desenho que valem explicitar:

**Orçamento de compute fixo em passos, não em épocas.** Com o `TwoStreamBatchSampler`, uma "época"
é `len(rotulados) // labeled_batch_size` e encolhe conforme o orçamento de anotação. Fixar
`max_steps` garante que todo run vê o mesmo número de imagens — e é também o que ancora o
cronograma de consistência (`consistency_*_fraction`, ver `src/models/ssl/base.py`).

**Composição de batch idêntica em todos os orçamentos.** Todo batch tem
`LABELED_BATCH_SIZE` linhas rotuladas e `BATCH_SIZE - LABELED_BATCH_SIZE` não rotuladas, inclusive
em 100%. Sem o pool, o orçamento de 100% cairia num DataLoader comum com 8 linhas rotuladas por
passo — o dobro de gradiente supervisionado dos demais — e a margem medida sairia inflada por
construção.

**Nenhuma célula é degenerada.** Como o fluxo não rotulado é real e independe de `label_fraction`,
FixMatch e Mean Teacher têm um termo de consistência ativo também em 100%. A célula
`(método SSL, 100%)` é justamente o resultado que interessa reportar: o que os frames não anotados
somam sobre *toda* a anotação disponível.

In [ ]:
import torch
from omegaconf import OmegaConf

from src.experiment import Experiment
from src.evaluation import evaluate_per_class, aggregate_per_class

METHOD_CONFIG = {method: CONFIG_DIR / f"x1-binario-unet-{method}.yaml" for method in
                 ["supervised", "fixmatch", "meanteacher", "diffrect"]}

def run_id(method: str, fraction: float, seed: int) -> str:
    return f"{method}-lf{int(round(fraction * 100)):03d}-s{seed}"

In [ ]:
def steps_per_epoch(fraction: float) -> int:
    """Batches por epoca sob o regime dado -- define de quantas em quantas epocas validar.

    Com o pool nao rotulado ativo, o TwoStreamBatchSampler e usado em TODOS os orcamentos (inclusive
    100%), entao uma epoca e sempre uma passada pelos rotulos revelados.
    """
    n_labeled = int(regime_summary.loc[f"{fraction:.0%}", "rotulos revelados"])
    return max(1, n_labeled // LABELED_BATCH_SIZE)


def build_overrides(method: str, fraction: float, seed: int) -> dict:
    """Sobrescritas aplicadas por cima do YAML do metodo (OmegaConf.merge, aninhado)."""
    check_val_every_n_epoch = max(1, round(VAL_EVERY_N_STEPS / steps_per_epoch(fraction)))
    return {
        "name": f"x1-binario-{run_id(method, fraction, seed)}",
        "trainer": {"params": {
            "max_steps": MAX_STEPS,
            "check_val_every_n_epoch": check_val_every_n_epoch,
        }},
        "lr_scheduler": {"params": {"T_max": MAX_STEPS}},
        "data": {"params": {
            "batch_size": BATCH_SIZE,
            "shared_dataset_params": {
                "label_fraction": fraction,
                "label_seed": LABEL_SEED,
                "label_group_column": LABEL_GROUP_COLUMN,
                "split_seed": SPLIT_SEED,
                "split_group_column": SPLIT_GROUP_COLUMN,
                # Pool identico em todos os orcamentos: e a variavel que NAO varia.
                "unlabeled_pool_size": UNLABELED_POOL_SIZE,
                "unlabeled_pool_frame_stride": UNLABELED_POOL_FRAME_STRIDE,
                "unlabeled_pool_seed": UNLABELED_POOL_SEED,
                "unlabeled_pool_splits": UNLABELED_POOL_SPLITS,
                "expose_hidden_targets": True,   # diagnostico de pseudo-rotulo; nenhuma loss le
            },
            "train_batch_sampler": {"params": {
                "batch_size": BATCH_SIZE,
                "labeled_batch_size": LABELED_BATCH_SIZE,
                "seed": seed,
                "unlabeled_policy": UNLABELED_POLICY,   # variado apenas no X1B
            }},
        }},
    }

pd.DataFrame([
    {"orcamento": f"{f:.0%}", "passos/epoca": steps_per_epoch(f),
     "epocas ate MAX_STEPS": MAX_STEPS // steps_per_epoch(f),
     "valida a cada N epocas": max(1, round(VAL_EVERY_N_STEPS / steps_per_epoch(f)))}
    for f in LABEL_FRACTIONS
]).set_index("orcamento")

### O runner

Grava incrementalmente em `logs/x1-binario/results_per_sample.csv` e pula toda célula já presente no
arquivo — então o loop é **retomável**: se a sessão cair no meio, basta re-executar a célula.

> ⚠️ Resultados gravados sob o desenho anterior (sem pool não rotulado, orçamentos 10/20/50/100)
> **não são comparáveis** com estes. Apague `results_per_sample.csv` antes do primeiro run.

In [ ]:
def load_results() -> pd.DataFrame:
    if RESULTS_CSV.exists():
        return pd.read_csv(RESULTS_CSV)
    return pd.DataFrame()


def save_results(results: pd.DataFrame) -> None:
    results.to_csv(RESULTS_CSV, index=False)


def train_and_evaluate(method: str, fraction: float, seed: int) -> pd.DataFrame:
    """Treina uma celula da grade e devolve as metricas por amostra do test set."""
    experiment = Experiment(
        config_path=str(METHOD_CONFIG[method]),
        overrides=build_overrides(method, fraction, seed),
        log_dir=LOGS_DIR,
    )
    experiment.run(seed=seed)

    # O Mean Teacher reporta pelo teacher (os pesos EMA sao o modelo melhor); os demais pelo proprio
    # modelo. `_eval_forward` existe so no wrapper do Mean Teacher.
    forward_fn = getattr(experiment.lit_module, "_eval_forward", None)

    per_sample = evaluate_per_class(
        experiment.lit_module,
        experiment.data_module.test_dataloader(),
        num_classes=NUM_CLASSES,
        class_ids=CLASS_IDS,
        forward_fn=forward_fn,
    )
    per_sample["output_dir"] = str(experiment.experiment_run_dir)
    return per_sample

In [ ]:
results = load_results()

for seed in SEEDS:
    for fraction in LABEL_FRACTIONS:
        for method in METHODS:
            rid = run_id(method, fraction, seed)

            if len(results) and (results.run_id == rid).any():
                print(f"[pula] {rid} (ja em {RESULTS_CSV.name})")
                continue

            print(f"[treina] {rid}")
            per_sample = train_and_evaluate(method, fraction, seed)

            per_sample.insert(0, "run_id", rid)
            per_sample.insert(1, "method", method)
            per_sample["label_fraction"] = fraction
            per_sample["seed"] = seed

            results = pd.concat([results, per_sample], ignore_index=True)
            save_results(results)

print(f"\nConcluido: {results.run_id.nunique()} runs, {len(results)} linhas em {RESULTS_CSV}")

---

# Parte 3 — Leitura dos resultados

Duas perguntas distintas saem da mesma grade:

1. **Quanto vale anotar mais?** A inclinação da curva do supervisionado entre 25% e 100%. Plana ⇒
   anotar mais frames não compra desempenho neste dataset.
2. **Quanto valem os frames não anotados?** A distância vertical entre a curva de cada método SSL e
   a do supervisionado, *no mesmo orçamento*. A leitura mais importante é em **100%**: ali o
   supervisionado já viu toda a anotação existente, e o que o método SSL adicionar vem
   exclusivamente dos frames nunca anotados.

### Uma única classe reportada: `FG`

Como o modelo já é treinado com `n_classes=2`, não há `C2`/`C4` separados aqui para fundir — a tabela e os gráficos abaixo têm uma única coluna/classe, `FG`.

A comparação com o X1-multiclasse é `FG` (aqui) vs. `C2+C4 (macro)` ([experiment_1_does_ssl_helps.ipynb](experiment_1_does_ssl_helps.ipynb)) — mas leia com a ressalva do topo deste notebook: a macro é sensível a confusão C2↔C4 e `FG` não tem como sofrer dessa confusão (só existe uma estrutura aqui). Uma diferença entre as duas mistura o efeito de treinar binário desde o início com o efeito de nunca ter havido confusão entre duas classes — não isole um sem o outro.

In [ ]:
results = load_results()

aggregated = aggregate_per_class(results, group_columns=["method", "label_fraction", "seed"])
# Media entre sementes (com uma so semente, e a propria).
across_seeds = (
    aggregated
    .groupby(["method", "label_fraction", "class"], as_index=False)[METRICS]
    .mean()
)

table = across_seeds.pivot_table(index=["method", "label_fraction"], columns="class", values=METRICS)
table = table.reorder_levels([1, 0], axis=1).sort_index(axis=1, level=0)
table.round(4)

### As curvas

A linha preta é o supervisionado: sua inclinação responde a pergunta 1. A distância das outras
linhas até ela, em cada x, responde a pergunta 2.

In [ ]:
class_names = list(CLASS_IDS.values())
fig, axes = plt.subplots(len(METRICS), len(class_names), figsize=(5 * len(class_names), 3.0 * len(METRICS)), squeeze=False)
colors = {"supervised": "#333333", "fixmatch": "#4C72B0", "meanteacher": "#DD8452", "diffrect": "#55A868"}

for row, metric in enumerate(METRICS):
    for col, class_name in enumerate(class_names):
        ax = axes[row][col]
        subset = across_seeds[across_seeds["class"] == class_name]
        for method in METHODS:
            data = subset[subset.method == method].sort_values("label_fraction")
            if data.empty:
                continue
            ax.plot(data.label_fraction * 100, data[metric], "o-",
                    label=method, color=colors.get(method), linewidth=2 if method == "supervised" else 1.4)
        ax.set_xticks([f * 100 for f in LABEL_FRACTIONS])
        ax.set_xlabel("orcamento de anotacao (%)")
        ax.set_ylabel(metric)
        ax.set_title(f"{metric.upper()} - {class_name}" + ("  (menor e melhor)" if metric in ("assd", "hd95") else ""))
        ax.spines[["top", "right"]].set_visible(False)
        if row == 0 and col == 0:
            ax.legend(frameon=False, fontsize=8)

plt.tight_layout()

### Grade método × orçamento (barras)

Linha = orçamento (100% no topo, 25% na base), coluna = métrica; cada célula é um gráfico de barras comparando os métodos naquele orçamento/métrica, para a única classe reportada (`FG`).

In [ ]:
method_colors = {"supervised": "#333333", "fixmatch": "#4C72B0", "meanteacher": "#DD8452", "diffrect": "#55A868"}
fractions_desc = sorted(LABEL_FRACTIONS, reverse=True)


def bar_grid(data: pd.DataFrame, title: str):
    """Grade orcamento (linha, 100% no topo) x metrica (coluna); cada celula e um bar chart por metodo."""
    fig, axes = plt.subplots(len(fractions_desc), len(METRICS),
                              figsize=(4.0 * len(METRICS), 3.0 * len(fractions_desc)), squeeze=False)

    for row, fraction in enumerate(fractions_desc):
        subset = data[data.label_fraction == fraction]
        for col, metric in enumerate(METRICS):
            ax = axes[row][col]
            present = [m for m in METHODS if not subset[subset.method == m].empty]
            values = [subset.loc[subset.method == m, metric].iloc[0] for m in present]
            ax.bar(present, values, color=[method_colors[m] for m in present])
            # set_xticks ANTES de set_xticklabels: sem isso o matplotlib usa um locator nao-fixo
            # e emite "UserWarning: set_ticklabels() should only be used with ... FixedLocator"
            # uma vez por eixo.
            ax.set_xticks(range(len(present)))
            ax.set_xticklabels(present, rotation=30, ha="right")
            ax.spines[["top", "right"]].set_visible(False)
            if row == 0:
                ax.set_title(metric.upper() + ("\n(menor e melhor)" if metric in ("assd", "hd95") else ""))
            if col == 0:
                ax.set_ylabel(f"{fraction:.0%}")

    fig.suptitle(title, y=1.02, fontsize=13)
    plt.tight_layout()
    # Fecha a figura para o backend inline nao mostra-la de novo no fim da celula -- ela ja e
    # exibida uma vez pelo valor de retorno sendo a ultima expressao da celula. Sem isso o
    # grafico aparece duplicado (uma vez pelo hook de auto-show, outra pelo display do retorno).
    plt.close(fig)
    return fig


bar_grid(across_seeds[across_seeds["class"] == "FG"], "Método × orçamento — FG")

### Os dois números

**Margem de anotação** — o que o supervisionado ganha ao sair do menor orçamento para 100%. É o
teto do que qualquer método poderia recuperar *escondendo menos rótulos*.

**Ganho do não anotado em 100%** — o que cada método SSL ganha sobre o supervisionado treinado com
todos os rótulos. Este é o número publicável: não existe rótulo escondido nessa célula, então o
ganho só pode vir do pool.

In [ ]:
supervised = across_seeds[across_seeds.method == "supervised"]
class_names = list(CLASS_IDS.values())
low, high = min(LABEL_FRACTIONS), max(LABEL_FRACTIONS)

# --- numero 1: quanto vale anotar mais ---------------------------------------------------------
gate_rows = []
for class_name in class_names:
    data = supervised[supervised["class"] == class_name].set_index("label_fraction")
    row = {"classe": class_name}
    for metric in METRICS:
        v_low, v_high = data.loc[low, metric], data.loc[high, metric]
        # ASSD/HD95 melhoram descendo; Dice/IoU melhoram subindo.
        gain = (v_high - v_low) if metric in ("dice", "iou") else (v_low - v_high)
        row[f"{metric} @{low:.0%}"] = round(v_low, 4)
        row[f"{metric} @{high:.0%}"] = round(v_high, 4)
        row[f"ganho {metric}"] = round(gain, 4)
    gate_rows.append(row)

gate = pd.DataFrame(gate_rows).set_index("classe")
display(gate[[c for c in gate.columns if c.startswith("ganho")] +
             [c for c in gate.columns if not c.startswith("ganho")]])

print(f"Margem de anotacao (supervisionado {high:.0%} - supervisionado {low:.0%}):")
for class_name in class_names:
    print(f"  {class_name}: {gate.loc[class_name, 'ganho dice']:+.4f} Dice")

# --- numero 2: quanto valem os frames nunca anotados --------------------------------------------
# Medido em 100% dos rotulos: nao ha rotulo escondido nessa celula, entao qualquer ganho vem do pool.
print(f"{chr(10)}Ganho dos frames nao anotados, com {high:.0%} dos rotulos (SSL - supervisionado):")
pool_rows = []
for class_name in class_names:
    baseline = supervised[(supervised["class"] == class_name) &
                          (supervised.label_fraction == high)]["dice"]
    if baseline.empty:
        continue
    baseline = baseline.iloc[0]
    for method in [m for m in METHODS if m != "supervised"]:
        data = across_seeds[(across_seeds.method == method) &
                            (across_seeds["class"] == class_name) &
                            (across_seeds.label_fraction == high)]["dice"]
        if data.empty:
            continue
        pool_rows.append({"classe": class_name, "metodo": method,
                          "dice": round(data.iloc[0], 4),
                          "supervisionado 100%": round(baseline, 4),
                          "ganho dice": round(data.iloc[0] - baseline, 4)})
        print(f"  {class_name} / {method}: {data.iloc[0] - baseline:+.4f} Dice")

pd.DataFrame(pool_rows).set_index(["classe", "metodo"]) if pool_rows else "sem runs SSL em 100% ainda"

### Quanto da margem cada método recupera

Fração da margem do supervisionado que o método fecha, em cada orçamento. `100%` significaria
igualar o supervisionado treinado com todos os rótulos; valores negativos significam que o método
**piorou** em relação a treinar só com os rótulos revelados — resultado comum e informativo.

In [ ]:
# Em cada orcamento < 100%, quanto da margem de anotacao o metodo recupera sem ver os rotulos
# escondidos. 100% significaria igualar o supervisionado treinado com todos os rotulos; negativo
# significa que o metodo PIOROU em relacao a treinar so com os rotulos revelados -- resultado comum
# e informativo (confirmation bias). Cruze com `train/pseudo_accuracy` no TensorBoard antes de
# concluir que "o metodo nao funciona".
recovery_rows = []
for class_name in class_names:
    ceiling = supervised[(supervised["class"] == class_name) &
                         (supervised.label_fraction == high)]["dice"].iloc[0]
    for fraction in [f for f in LABEL_FRACTIONS if f < high]:
        floor = supervised[(supervised["class"] == class_name) &
                           (supervised.label_fraction == fraction)]["dice"].iloc[0]
        margin = ceiling - floor
        for method in [m for m in METHODS if m != "supervised"]:
            data = across_seeds[(across_seeds.method == method) &
                                (across_seeds["class"] == class_name) &
                                (across_seeds.label_fraction == fraction)]
            if data.empty:
                continue
            recovered = data["dice"].iloc[0] - floor
            recovery_rows.append({
                "classe": class_name,
                "orcamento": f"{fraction:.0%}",
                "metodo": method,
                "dice": round(data["dice"].iloc[0], 4),
                "sup. no mesmo orcamento": round(floor, 4),
                "margem disponivel": round(margin, 4),
                "% da margem recuperada": round(100 * recovered / margin, 1) if abs(margin) > 1e-6 else np.nan,
            })

pd.DataFrame(recovery_rows).set_index(["classe", "orcamento", "metodo"]) if recovery_rows else "sem runs SSL ainda"

---

## Notas de reprodutibilidade e fase 2

**Cronograma de consistência.** `consistency_rampup_fraction` / `consistency_warmup_fraction` são
frações do orçamento (`max_steps`), não passos absolutos. O SSL4MIS embute `iter_num // 150` contra
`--consistency_rampup 200.0` porque 150×200 = 30000 = seu `max_iterations`: o divisor existe para a
rampa completar no último passo. Copiar 150/200 para um run de 8000 passos faria o peso de
consistência parar em ~0.7% do configurado — os métodos SSL sairiam idênticos ao supervisionado por
aritmética, não por dados. Ver `src/models/ssl/base.py`.

**O que acompanhar durante o treino** (TensorBoard em `logs/x1-binario-*/<timestamp>/tensorboard`):

| Curva | Lê o quê |
|---|---|
| `train/sup_loss`, `train/unsup_loss`, `train/unsup_loss_weighted` | os componentes da loss separados |
| `train/consistency_weight_step` | a rampa de fato subindo até `consistency_weight` |
| `train/labeled_in_batch`, `train/unlabeled_in_batch` | a composição do batch (deve ser 4/4 sempre) |
| `train/pseudo_accuracy`, `train/pseudo_dice`, `train/pseudo_label_coverage` | qualidade do pseudo-rótulo do FixMatch contra a GT escondida |
| `train/teacher_accuracy`, `train/teacher_dice` | qualidade do alvo do Mean Teacher |
| `train/pseudo_rows_with_gt` | quantas linhas não rotuladas do batch tinham GT para pontuar |

Cobertura subindo com acurácia caindo é *confirmation bias* acontecendo em tempo real — a razão
mais comum de um método semi-supervisionado ficar abaixo da sua própria baseline. Esses tensores
vêm de `expose_hidden_targets=True` e **nenhuma loss os lê**; frames do pool não têm anotação e são
excluídos da conta (`has_hidden_target=False`).

**Proveniência dos métodos.** Os wrappers vivem em `src/models/ssl/`. Mean Teacher é uma porta fiel
de `train_mean_teacher_2D.py` do [SSL4MIS](https://github.com/HiLab-git/SSL4MIS) (MIT) — mesma
composição de loss (`0.5*(CE+Dice)`), mesmo ruído no teacher, mesma forma de rampa. **FixMatch
diverge de propósito:** o do SSL4MIS não é o canônico (adiciona uma *complementary loss* ponderada
por entropia), e o X1 quer a baseline mais simples possível, então implementamos o FixMatch do paper
(limiar de confiança + CE no pseudo-rótulo). Cada divergência está registrada na docstring do módulo
correspondente, e `tests/test_reference_equivalence.py` prende a fidelidade numérica.

**DiffRect** é um slot registrado (`src/models/ssl/diffrect.py`) que ainda levanta
`NotImplementedError`. Para ativá-lo: portar os módulos LCC + difusão latente de
[CUHK-AIM-Group/DiffRect](https://github.com/CUHK-AIM-Group/DiffRect) (MIT) e adicionar
`"diffrect"` a `METHODS`.

**Experimento X1B.** A política de amostragem dos frames não rotulados
(`train_batch_sampler.params.unlabeled_policy`) é fixada aqui em `same_video` e variada em
[experiment_1b_unlabeled_sampling_policy.ipynb](experiment_1b_unlabeled_sampling_policy.ipynb), que
compara `global`, `same_video` e janelas temporais de W = 3, 5 e 10 frames. Se o X1B mostrar que
`global` é consistentemente melhor, o default deste notebook precisa ser revisto e as conclusões
relidas.

**Para aumentar a robustez do resultado**, em ordem de custo:
1. mais sementes em `SEEDS` (mede a variância de treino — com uma só não há barra de erro);
2. `LABEL_GROUP_COLUMN = "video_id"` (checa se a conclusão depende de *quais* pacientes caíram);
3. variar `UNLABELED_POOL_SIZE` / `UNLABELED_POOL_FRAME_STRIDE` (o pool tem retorno decrescente:
   vale saber onde ele satura);
4. mais folds, variando `SPLIT_SEED` (mede a variância de split — o mais caro e o mais informativo).